In [203]:
from pathlib import Path
import pandas as pd

project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

sample_sheet_path = project_root / "metadata" / "sample_sheet.csv"

df = pd.read_csv(sample_sheet_path)

print(f"Loaded {len(df)} samples")

Loaded 17 samples


In [ ]:
#Cell 1: Creates conditions and new_rows

dtag_dosage_conditions = [
    ("CTRL", 0, 1),
    ("0p5nM", 0.5, 0.75),
    ("1nM", 1, 0.50),
    ("5nM", 5, 0.25),
    ("50nM", 50, 0),
]

flow_measurements = [
    (1, 0.7211394, 0.4957746, 0.2053973, 0.0314843),
    (1, 0.8112676, None, 0.156338, 0.0380282),
    (1, 0.823109, 0.4793153, 0.1540656, 0.0299572),
]

#reading from Nanodrop (ng/ul)
rna_conc = [ 
    (579.109, 566.892, 442.28,576.79, 549.082),
    (439.592, 410.552, 459.199, 466.334, 438.539),
    (422.983, 453.871, 461.877, 463.603, 453.945),
]

new_rows = []

for replicate in range(1, 4):
    for condition_name, concentration, nominal_dosage_pct in dtag_dosage_conditions:


        sample_id = f"B2_{condition_name}_r{replicate
}"
        # sample number = 0-14
        sample_num = (replicate - 1) * len(dtag_dosage_conditions) + dtag_dosage_conditions.index((condition_name, concentration, nominal_dosage_pct))
        # condition number = 0-4 (0=1, 1=0.75, 2=0.50, 3=0.25, 4=0)
        condition_num = dtag_dosage_conditions.index((condition_name, concentration, nominal_dosage_pct))

        row = {
            # --- identifiers ---
            "sample_id": sample_id,
            "count_column_name": sample_id,
            "fastq_prefix": "NS.X0338.004.UDP0" 
                 + str(273 + sample_num) 
                 + "_i7",
            # --- dosage ---
            "dtag_conc_nM": concentration,
            "nominal_dosage_pct": nominal_dosage_pct,
            # --- flow cytometry / mNeonGreen ---
            "mng_measured_pct": flow_measurements[replicate - 1][condition_num],
            "mng_summary_stat": "Median",
            "mng_autofluor_corrected": 1,
            "fcs_file": "B2" + f"_{concentration}" + "nM_rep" + f"{replicate}" + ".fcs",
            "flow_reference_sample": "WT_0nM_rep" + f"{replicate}" + ".fcs",
            # --- replicate / culture history ---
            "replicate": replicate,
            "differentiation_batch": None,
            "differentiation_date": None,
            "treatment_date": (
                "2026-06-24"
                if replicate == 1 and condition_num != 2
                else "2026-07-01"
            ),
            "harvest_date": (
                "2026-06-26"
                if replicate == 1 and condition_num != 2
                else "2026-07-03"
            ),
            "passage_number": (
                "P7"
                if replicate == 1 and condition_num != 2
                else "P8"
            ),
            "cells_harvested": None,
            # --- RNA / library prep ---
            "rna_yield_ng": rna_conc[replicate - 1][condition_num] * 60, #concentration (ng/ul) * volume submitted (ul)
            "rin": None,
            "spike_in_added": None,
            "spike_in_amount": None,
            "library_prep_batch": 1,
            "library_prep_kit": "polyA",
            # --- sequencing ---
            "sequencing_run": None,
            "lane": None,
            # --- free text ---
            "notes": None,
        }

        new_rows.append(row)

In [205]:
# Reload the latest sample sheet
df = pd.read_csv(sample_sheet_path)

# Convert generated rows into a DataFrame
new_df = pd.DataFrame(new_rows)

# Remove duplicate generated IDs
new_df = new_df.drop_duplicates(
    subset="sample_id",
    keep="last"
)

# Match the sample-sheet columns
new_df = new_df.reindex(columns=df.columns)

# Remove existing versions of these samples
new_sample_ids = new_df["sample_id"]
df = df[~df["sample_id"].isin(new_sample_ids)]

# Add the new versions
df = pd.concat([df, new_df], ignore_index=True)

# Final duplicate protection
df = df.drop_duplicates(
    subset="sample_id",
    keep="last"
)

# Sort the samples
df["dtag_conc_nM"] = pd.to_numeric(
    df["dtag_conc_nM"],
    errors="coerce"
)

df["replicate"] = pd.to_numeric(
    df["replicate"],
    errors="coerce"
)

df = df.sort_values(
    ["dtag_conc_nM", "replicate"]
).reset_index(drop=True)

# Save
df.to_csv(sample_sheet_path, index=False)

print(f"Saved {len(df)} unique samples")
print(f"Duplicate IDs: {df['sample_id'].duplicated().sum()}")

df

Saved 17 unique samples
Duplicate IDs: 0


,sample_id,count_column_name,fastq_prefix,dtag_conc_nM,nominal_dosage_pct,mng_measured_pct,mng_summary_stat,mng_autofluor_corrected,fcs_file,flow_reference_sample,...,cells_harvested,rna_yield_ng,rin,spike_in_added,spike_in_amount,library_prep_batch,library_prep_kit,sequencing_run,lane,notes
0,EXAMPLE_DELETE_THIS_ROW,B2_DMSO_r1,B2_DMSO_r1_S1,0.0,100.00,100.000000,median,yes,B2_DMSO_r1.fcs,B2_DMSO_r1,...,1200000.0,850.00,9.4,no,NaN,prep_1,polyA,NS500_run17,1.0,reference sample for both flow and expression
1,B2_CTRL_r1,B2_CTRL_r1,NS.X0338.004.UDP0273_i7,0.0,1.00,1.000000,Median,1,B2_0nM_rep1.fcs,WT_0nM_rep1.fcs,...,None,34746.54,None,None,None,None,polyA,None,None,None
2,B2_CTRL_r2,B2_CTRL_r2,NS.X0338.004.UDP0278_i7,0.0,1.00,1.000000,Median,1,B2_0nM_rep2.fcs,WT_0nM_rep2.fcs,...,None,26375.52,None,None,None,None,polyA,None,None,None
3,B2_CTRL_r3,B2_CTRL_r3,NS.X0338.004.UDP0283_i7,0.0,1.00,1.000000,Median,1,B2_0nM_rep3.fcs,WT_0nM_rep3.fcs,...,None,25378.98,None,None,None,None,polyA,None,None,None
4,B2_0p5nM_r1,B2_0p5nM_r1,NS.X0338.004.UDP0274_i7,0.5,0.75,0.721139,Median,1,B2_0.5nM_rep1.fcs,WT_0nM_rep1.fcs,...,None,34013.52,None,None,None,None,polyA,None,None,None
5,B2_0p5nM_r2,B2_0p5nM_r2,NS.X0338.004.UDP0279_i7,0.5,0.75,0.811268,Median,1,B2_0.5nM_rep2.fcs,WT_0nM_rep2.fcs,...,None,24633.12,None,None,None,None,polyA,None,None,None
6,B2_0p5nM_r3,B2_0p5nM_r3,NS.X0338.004.UDP0284_i7,0.5,0.75,0.823109,Median,1,B2_0.5nM_rep3.fcs,WT_0nM_rep3.fcs,...,None,27232.26,None,None,None,None,polyA,None,None,None
7,B2_1nM_r1,B2_1nM_r1,NS.X0338.004.UDP0275_i7,1.0,0.50,0.495775,Median,1,B2_1nM_rep1.fcs,WT_0nM_rep1.fcs,...,None,26536.80,None,None,None,None,polyA,None,None,None
8,B2_1nM_r2,B2_1nM_r2,NS.X0338.004.UDP0280_i7,1.0,0.50,NaN,Median,1,B2_1nM_rep2.fcs,WT_0nM_rep2.fcs,...,None,27551.94,None,None,None,None,polyA,None,None,None
9,B2_1nM_r3,B2_1nM_r3,NS.X0338.004.UDP0285_i7,1.0,0.50,0.479315,Median,1,B2_1nM_rep3.fcs,WT_0nM_rep3.fcs,...,None,27712.62,None,None,None,None,polyA,None,None,None
